# 00 - Reproduce the paper's TMD baseline

**Run this first.** Every improvement notebook (01-04) is judged against the paper's Table 1, and that comparison is only meaningful if *our* pipeline reproduces the paper's own TMD numbers under the same protocol.

Order of work, cheapest decisive check first:
1. fine-tune BERT with the **authors' recipe** (10 epochs, batch 64, keep the final epoch; `T.PAPER_FT`);
2. attack the **vanilla** model right away. No GAN is involved, so this isolates the fine-tuned model and the attack from the defense. Paper: PWWS 39.2, TextFooler 27.9 (our earlier 3-epoch model gave 27.5 and 23.5);
3. InfoGAN hyperparameter search (best of 8 by validation reconstruction loss, as the authors did);
4. a cheap diagnostic of the projection, then the TMD attacks and the comparison table.

Everything is cached on Google Drive under `WORK` and resumes after a disconnect (fine-tuning per epoch, attacks per 50 examples).

**Known differences from the paper:** pip `textattack` instead of the authors' fork (constraints and the PWWS search rebuilt to match theirs); one learning rate instead of their grid; noise of about +-7 points per attack cell at `N_EVAL=200` (+-3 at 1000). This code has not been run by its author.

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)
# If pip reports a numpy / protobuf conflict, restart the runtime once and re-run from here.

In [ ]:
import torch, tmd_lab as T
T.setup_env()
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'   # 'ag_news' | 'imdb' | 'yelp_polarity'
N_EVAL = 200          # attacked examples per attack (paper: 1000). 200 is enough to see whether the reproduction works
BASE_K = 30           # sampling size k of the paper's projection (repo default)
GAN_SEARCH_TRIALS = 8   # InfoGANs to train; the best by validation reconstruction loss is kept (0 = a single GAN)
ATTACKS = ('pwws', 'textfooler')   # add 'bertattack' last: it is by far the slowest (hours on a T4)
WORK = '/content/drive/MyDrive/tmd_run_paperft'   # shared by notebooks 00-04 (new folder: authors' fine-tuning recipe)

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base', **T.PAPER_FT)   # 10 epochs, batch 64, final epoch kept
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
E_te, Y_te = lab.embeddings('test', 'base')

In [ ]:
# Step 2: attack the vanilla model first (no GAN involved). Paper: PWWS 39.2, TextFooler 27.9, clean 94.39.
vanilla = T.Victim(lab, mode='none')
res_vanilla = T.evaluate(lab, vanilla, 'vanilla', E_te, Y_te, n=N_EVAL, attacks=ATTACKS)
paper = T.PAPER[DATASET]['vanilla']
for i, m in enumerate(T.METRICS):
    if m in res_vanilla:
        print(f'{T.METRIC_LABEL[m]:12s} ours {res_vanilla[m]["acc"]:5.1f}   paper {paper[i]:5.1f}')
print('If vanilla is now close to the paper, the fine-tuning recipe was the main gap. If not, tell Claude before continuing.')

In [ ]:
# Step 3: InfoGAN (best of GAN_SEARCH_TRIALS by validation reconstruction loss)
gan = lab.get_gan('base', E_tr, E_va)
if GAN_SEARCH_TRIALS:   # the authors kept the best of a random hyperparameter search (by validation loss)
    gan, hp = T.search_gans(lab, 'base', E_tr, E_va, trials=GAN_SEARCH_TRIALS)

In [ ]:
# Diagnostic (no attacks, takes seconds): is the generator collapsed to roughly one prototype per cluster?
Et = E_te[:2000].to(T.DEV)
with torch.no_grad():
    outs = [T.project(gan, Et, k=BASE_K)[0] for _ in range(10)]               # 10 independent projections
    preds = torch.stack([lab.model.classifier(o).argmax(-1) for o in outs])
    flip = (preds != preds[0]).any(0).float().mean().item()
    d_proj = torch.norm(outs[0] - Et, dim=-1).mean().item()
    d_draws = torch.norm(outs[0] - outs[1], dim=-1).mean().item()
    d_real = torch.norm(Et - Et[torch.randperm(len(Et), device=T.DEV)], dim=-1).mean().item()
    cte = gan.dis(Et)[1].argmax(-1).cpu()
    ctr = torch.cat([gan.dis(E_tr[i:i + 4096].to(T.DEV))[1].argmax(-1).cpu() for i in range(0, len(E_tr), 4096)])
maj = torch.stack([torch.bincount(Y_tr[ctr == k], minlength=lab.cfg['n_cls']).argmax() if (ctr == k).any()
                   else torch.tensor(0) for k in range(gan.code_dim)])
print(f'prediction changes across 10 projections for {100 * flip:.1f}% of texts')
print(f'||t_hat - t|| = {d_proj:.3f} | between two draws of the same text = {d_draws:.3f} | between random real texts = {d_real:.3f}')
print('clusters used:', len(cte.unique()), 'of', gan.code_dim)
print('accuracy of cluster -> majority label on test:', round((maj[cte] == Y_te[:2000]).float().mean().item() * 100, 1))


In [ ]:
base = T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K)
res_base = T.evaluate(lab, base, 'baseline', E_te, Y_te, n=N_EVAL, attacks=ATTACKS)

In [ ]:
import pandas as pd
paper = T.PAPER[DATASET]
rows = []
for i, m in enumerate(T.METRICS):
    if m not in res_vanilla or m not in res_base: continue   # attacks not run yet
    rows.append({'metric': T.METRIC_LABEL[m],
                 'paper vanilla': paper['vanilla'][i], 'our vanilla': round(res_vanilla[m]['acc'], 1),
                 'paper TMD': paper['TMD'][i], 'our TMD (baseline)': round(res_base[m]['acc'], 1)})
display(pd.DataFrame(rows))
print('Reproduction check: our TMD should be close to the paper TMD column, and clearly above our vanilla.')
print('Per-cell noise at n=1000 is about +-3 points for the attack columns.')
display(T.compare(lab, 'baseline'))